# Master's Thesis Notebook

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026


# test3 spectral

Notebook equivalent of `tests/1_core/test3_spectral.py`. The original Python test module remains the source of truth; this notebook imports it and runs each `test_*` function in its own cell for interactive inspection.

The notebook uses the shared TPeanuts output-root convention defined in section `2. Paths`. Generated test artifacts and figures are redirected to the notebook-specific `OUTPUT_DIR` under `OUTPUT_TEST_ROOT / "core" / NOTEBOOK_STEM`.


## 1. Libraries and test module

This section locates the repository, imports the original Python test module, and reloads it so changes in the source file are reflected when the notebook is rerun. The expected result is that the module imports without errors. Possible problems include running the notebook from outside the repository tree or missing editable installation paths.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import os
import sys
import traceback
from pathlib import Path

import matplotlib.pyplot as plt


from tpeanuts.util.notebooks import find_repo_root
HERE = Path.cwd().resolve()
PACKAGE_DIR = find_repo_root(HERE, folder="analysis")
print(f"PACKAGE_DIR = {PACKAGE_DIR}")

NOTEBOOK_STEM = "test3_spectral"
TEST_MODULE_PATH = PACKAGE_DIR / "notebooks" / "tests" / "1_core" / f"{NOTEBOOK_STEM}.py"
TEST_MODULE_NAME = f"notebook_tests_{TEST_MODULE_PATH.parent.name.replace('.', '_')}_{NOTEBOOK_STEM}"
if not TEST_MODULE_PATH.exists():
    raise FileNotFoundError(f"Could not find local test module: {TEST_MODULE_PATH}")
spec = importlib.util.spec_from_file_location(TEST_MODULE_NAME, TEST_MODULE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load test module spec from {TEST_MODULE_PATH}")
TEST_MODULE = importlib.util.module_from_spec(spec)
sys.modules[TEST_MODULE_NAME] = TEST_MODULE
spec.loader.exec_module(TEST_MODULE)
print("Loaded local test module:", TEST_MODULE_PATH)

from tpeanuts.util.notebooks import build_notebook_test_runner


PACKAGE_DIR = G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts
Loaded local test module: G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\notebooks\tests\1_core\test3_spectral.py


## 2. Paths

All outputs are rooted at `DEFAULT_OUTPUT_ROOT` unless `TPEANUTS_OUTPUT_ROOT` is defined in the environment. This notebook writes test figures and any generated artifacts under `OUTPUT_TEST_ROOT / "core" / NOTEBOOK_STEM`, while data, analysis, benchmark, and external roots are defined with the same shared structure used by the analysis notebooks.


In [2]:
DEFAULT_OUTPUT_ROOT = Path(r"V:\output")
OUTPUT_ROOT = Path(os.environ.get("TPEANUTS_OUTPUT_ROOT", DEFAULT_OUTPUT_ROOT))

OUTPUT_DATA_ROOT = Path(OUTPUT_ROOT / "data")
OUTPUT_ANALYSIS_ROOT = Path(OUTPUT_ROOT / "analysis")
OUTPUT_BENCHMARK_ROOT = Path(OUTPUT_ROOT / "benchmark")
OUTPUT_TEST_ROOT = Path(OUTPUT_ROOT / "test")

OUTPUT_DATA_ATMOSPHERE = Path(OUTPUT_DATA_ROOT / "atmosphere")
OUTPUT_DATA_SOLAR = Path(OUTPUT_DATA_ROOT / "solar")
OUTPUT_DATA_EXTERNAL = Path(OUTPUT_DATA_ROOT / "external")

OUTPUT_ANALYSIS_ATMOSPHERE = Path(OUTPUT_ANALYSIS_ROOT / "atmosphere")
OUTPUT_ANALYSIS_SOLAR = Path(OUTPUT_ANALYSIS_ROOT / "solar")
OUTPUT_ANALYSIS_EXTERNAL = Path(OUTPUT_ANALYSIS_ROOT / "external")

OUTPUT_DATA_MCEQ = Path(OUTPUT_DATA_ROOT / "mceq")
OUTPUT_DATA_HONDA = Path(OUTPUT_DATA_ROOT / "honda")

OUTPUT_DIR = OUTPUT_TEST_ROOT / "core" / NOTEBOOK_STEM
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHOW_PLOTS = True
print("Output root     :", OUTPUT_ROOT)
print("Output directory:", OUTPUT_DIR)


Output root     : V:\output
Output directory: V:\output\test\core\test3_spectral


In [3]:
NOTEBOOK_RUNNER_ATTRS = {}
if "RUN_REAL_DATA_ANALYSIS" in globals():
    NOTEBOOK_RUNNER_ATTRS["RUN_REAL_mceq_run1_analysis"] = RUN_REAL_DATA_ANALYSIS

runner = build_notebook_test_runner(
    TEST_MODULE,
    OUTPUT_DIR,
    show_plots=SHOW_PLOTS,
    auto_save_figures=True,
    extra_module_attrs=NOTEBOOK_RUNNER_ATTRS,
)
run_notebook_test = runner.run_test
run_notebook_call = runner.run_call


## Test: `test_identity3_like`

**What is checked:** Identity3 like behavior and numerical consistency.
- H, *_ = build_hamiltonian()
- I = identity3_like(H)

**Expected result:** The expected result is a clean pass for these checks: assert_true; identity3_like correctness.
- I_expected = torch.eye(3, dtype=H.dtype, device=H.device)
  
**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [4]:
run_notebook_test(TEST_MODULE.test_identity3_like)


Running test_identity3_like ...

------------------------------------------------------------------------------------------
TEST: identity3_like
------------------------------------------------------------------------------------------
I:
tensor([[1.0000000000e+00+0.j, 0.0000000000e+00+0.j, 0.0000000000e+00+0.j],
        [0.0000000000e+00+0.j, 1.0000000000e+00+0.j, 0.0000000000e+00+0.j],
        [0.0000000000e+00+0.j, 0.0000000000e+00+0.j, 1.0000000000e+00+0.j]], dtype=torch.complex128)
------------------------------------------------------------------------------------------
PASSED: test_identity3_like


## Test: `test_hamiltonian_trace_from_ki_and_V`

**What is checked:** Hamiltonian trace from ki and v behavior and numerical consistency.
- H, ki, V, *_ = build_hamiltonian()
- trace_expected = torch.diagonal(H, dim1=-2, dim2=-1).sum(dim=-1)
- trace_from_ki = hamiltonian_trace_from_ki_and_V(ki, V)
  
**Expected result:** The expected result is a clean pass for these checks: Tr(H) = k1 + k2 + k3 + V.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [5]:
run_notebook_test(TEST_MODULE.test_hamiltonian_trace_from_ki_and_V)


Running test_hamiltonian_trace_from_ki_and_V ...

------------------------------------------------------------------------------------------
TEST: hamiltonian_trace_from_ki_and_V
------------------------------------------------------------------------------------------
trace from H      = (44.65947847609311+0j)
trace from ki,V   = (44.65947847609312+0j)
------------------------------------------------------------------------------------------
PASSED: test_hamiltonian_trace_from_ki_and_V


## Test: `test_traceless_hamiltonian`

**What is checked:** Traceless hamiltonian behavior and numerical consistency.
-  H, *_ = build_hamiltonian()
-  T, Tr_H = traceless_hamiltonian(H)

**Expected result:** The expected result is a clean pass for these checks: Trace of H; Trace of T is zero; H = T + Tr(H)/3 I.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [6]:
run_notebook_test(TEST_MODULE.test_traceless_hamiltonian)


Running test_traceless_hamiltonian ...

------------------------------------------------------------------------------------------
TEST: traceless_hamiltonian
------------------------------------------------------------------------------------------
H:
tensor([[4.0989630242e+00+0.j, 5.4754209922e-01+0.j, 5.9490881081e+00+0.j],
        [5.4754209922e-01+0.j, 8.2705632199e-01+0.j, -8.2752894562e-02+0.j],
        [5.9490881081e+00+0.j, -8.2752894562e-02+0.j, 3.9733459130e+01+0.j]], dtype=torch.complex128)
T:
tensor([[-1.0787529801e+01+0.j, 5.4754209922e-01+0.j, 5.9490881081e+00+0.j],
        [5.4754209922e-01+0.j, -1.4059436503e+01+0.j, -8.2752894562e-02+0.j],
        [5.9490881081e+00+0.j, -8.2752894562e-02+0.j, 2.4846966305e+01+0.j]], dtype=torch.complex128)
Tr(H) = (44.65947847609311+0j)
Tr(T) = 0j
------------------------------------------------------------------------------------------
PASSED: test_traceless_hamiltonian


## Test: `test_hermitize`

**What is checked:** Hermitize behavior and numerical consistency.
- H_herm = hermitize(H)

**Expected result:** The expected result is a clean pass for these checks: Hermitized matrix is Hermitian.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [7]:
run_notebook_test(TEST_MODULE.test_hermitize)


Running test_hermitize ...

------------------------------------------------------------------------------------------
TEST: hermitize
------------------------------------------------------------------------------------------
Hermitized H:
tensor([[4.0989630242e+00+0.0000000000e+00j, 5.4754259922e-01+1.0000000000e-06j, 5.9490881081e+00+0.0000000000e+00j],
        [5.4754259922e-01-1.0000000000e-06j, 8.2705632199e-01+0.0000000000e+00j, -8.2752894562e-02+0.0000000000e+00j],
        [5.9490881081e+00+0.0000000000e+00j, -8.2752894562e-02+0.0000000000e+00j, 3.9733459130e+01+0.0000000000e+00j]], dtype=torch.complex128)
max|H-H†| = 0.000000e+00
------------------------------------------------------------------------------------------
PASSED: test_hermitize


## Test: `test_traceless_invariant_c1`

**What is checked:** Traceless invariant c1 behavior and numerical consistency.
- c1 = traceless_invariant_c1(T)

**Expected result:** The expected result is a clean pass for these checks: c1 = -Tr(T^2)/2.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [8]:
run_notebook_test(TEST_MODULE.test_traceless_invariant_c1)


Running test_traceless_invariant_c1 ...

------------------------------------------------------------------------------------------
TEST: traceless_invariant_c1
------------------------------------------------------------------------------------------
c1          = (-501.40344397969255+0j)
c1 expected = (-501.40344397969255+0j)
------------------------------------------------------------------------------------------
PASSED: test_traceless_invariant_c1


## Test: `test_traceless_eigenvalues`

**What is checked:** Traceless eigenvalues behavior and numerical consistency.
- lam = traceless_eigenvalues(T)

**Expected result:** The expected result is a clean pass for these checks: assert_true; Eigenvalues match torch.linalg.eigvalsh; Sum of traceless eigenvalues is zero.
-     lam_expected = torch.linalg.eigvalsh(T).to(dtype=T.dtype)

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [9]:
run_notebook_test(TEST_MODULE.test_traceless_eigenvalues)


Running test_traceless_eigenvalues ...

------------------------------------------------------------------------------------------
TEST: traceless_eigenvalues
------------------------------------------------------------------------------------------
lambda:
tensor([-1.4185559215e+01+0.j, -1.1628354874e+01+0.j, 2.5813914090e+01+0.j], dtype=torch.complex128)
sum(lambda) = (-8.881784197001252e-15+0j)
------------------------------------------------------------------------------------------
PASSED: test_traceless_eigenvalues


## Test: `test_spectral_projectors`

**What is checked:** Spectral projectors behavior and numerical consistency.
- M, lam, c1 = spectral_projectors_traceless(T)

**Expected result:** The expected result is a clean pass for these checks: assert_true; Sum_a M_a = I; T = Sum_a lambda_a M_a; assert_close.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [10]:
run_notebook_test(TEST_MODULE.test_spectral_projectors)


Running test_spectral_projectors ...

------------------------------------------------------------------------------------------
TEST: spectral_projectors_traceless
------------------------------------------------------------------------------------------
lambda:
tensor([-1.4185559215e+01+0.j, -1.1628354874e+01+0.j, 2.5813914090e+01+0.j], dtype=torch.complex128)
c1:
tensor(-5.0140344398e+02+0.j, dtype=torch.complex128)
Projectors M shape: torch.Size([3, 3, 3])
------------------------------------------------------------------------------------------
PASSED: test_spectral_projectors


## Test: `test_hamiltonian_spectral_data`

**What is checked:** Hamiltonian spectral data behavior and numerical consistency.
- data = hamiltonian_spectral_data(H, trace_H=trace_H)

**Expected result:** The expected result is a clean pass for these checks: assert_true; H = Sum_a (lambda_a + Tr(H)/3) M_a; Spectral data projectors sum to identity.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change.


In [11]:
run_notebook_test(TEST_MODULE.test_hamiltonian_spectral_data)


Running test_hamiltonian_spectral_data ...

------------------------------------------------------------------------------------------
TEST: hamiltonian_spectral_data
------------------------------------------------------------------------------------------
Returned keys: ['T', 'trace', 'lam', 'c1', 'M']
T shape: torch.Size([3, 3])
trace shape: torch.Size([])
lambda shape: torch.Size([3])
c1 shape: torch.Size([])
M shape: torch.Size([3, 3, 3])
------------------------------------------------------------------------------------------
PASSED: test_hamiltonian_spectral_data


## Test: `test_batch_spectral_data`

**What is checked:** Batch spectral data behavior and numerical consistency.
- H, ki, V, L, zero_mask = reduced_hamiltonian_from_polynomial_density()
- data = hamiltonian_spectral_data(H, trace_H=trace_H)

**Expected result:** The expected result is a clean pass for these checks: assert_true; Batched projectors sum to identity; Batched H spectral reconstruction.

**Possible problems:** a failed assertion usually indicates a numerical regression or an unexpected API change; shape, dtype, or broadcasting changes may break downstream vectorized calls.


In [12]:
run_notebook_test(TEST_MODULE.test_batch_spectral_data)


Running test_batch_spectral_data ...

------------------------------------------------------------------------------------------
TEST: batched spectral data
------------------------------------------------------------------------------------------
H batch shape: torch.Size([3, 3, 3])
T batch shape: torch.Size([3, 3, 3])
lambda batch shape: torch.Size([3, 3])
M batch shape: torch.Size([3, 3, 3, 3])
------------------------------------------------------------------------------------------
PASSED: test_batch_spectral_data
